<a href="https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

###1. Ranked actions + reason codes

The playbook ranks content for human review rather than automatically changing or publishing content.

The highest-priority items are pages with meaningful existing visibility and observable signals that suggest a review opportunity, such as declining performance, stale content, weak click capture, or relatively thin content.

Each recommendation includes a reason code so that the reviewer can understand why the page was prioritized.

The ranking is decision-support only. A high-priority recommendation does not mean that a refresh or other action will definitely improve future performance.

In [ ]:
# Load the same dataset used in the previous ML assignments

import pandas as pd

url = "https://raw.githubusercontent.com/FlyRank-Internship/week1-flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.head())

Rows: 30000
Columns: 44
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ..

In [ ]:
# ML-10 — Ranked action queue

import numpy as np
import pandas as pd

queue = df.copy()

# -----------------------------
# 1. Build simple observable signals
# -----------------------------

# High visibility: top 25% of impressions
impressions_threshold = queue["impressions_90d"].quantile(0.75)

# Old content: top 25% of days since last update
update_age_threshold = queue["days_since_last_update"].quantile(0.75)

# Weak CTR: below the median among pages with impressions
ctr_threshold = queue.loc[
    queue["impressions_90d"] > 0, "ctr"
].median()

queue["high_visibility"] = (
    queue["impressions_90d"] >= impressions_threshold
)

queue["declining"] = (
    queue["trend_direction"] == "down"
)

queue["stale"] = (
    queue["days_since_last_update"] >= update_age_threshold
)

queue["weak_ctr"] = (
    (queue["ctr"] < ctr_threshold) &
    (queue["impressions_90d"] >= impressions_threshold)
)

# -----------------------------
# 2. Calculate action score
# -----------------------------

queue["baseline_score"] = (
    2 * queue["high_visibility"].astype(int)
    + 1 * queue["declining"].astype(int)
    + 1 * queue["stale"].astype(int)
    + 1 * queue["weak_ctr"].astype(int)
)

# -----------------------------
# 3. Assign reason codes
# -----------------------------

def get_reason(row):

    if row["baseline_score"] >= 4:
        return "high_priority_refresh"

    elif row["weak_ctr"]:
        return "high_visibility_low_ctr"

    elif row["declining"] and row["stale"]:
        return "declining_stale_content"

    elif row["declining"]:
        return "declining_performance"

    elif row["stale"]:
        return "stale_content_review"

    elif row["high_visibility"]:
        return "high_visibility_monitor"

    else:
        return "low_priority_monitor"


queue["reason_code"] = queue.apply(get_reason, axis=1)

# -----------------------------
# 4. Map reason codes to actions
# -----------------------------

action_map = {
    "high_priority_refresh": "Review and Refresh",
    "high_visibility_low_ctr": "Review Title / Snippet",
    "declining_stale_content": "Review and Refresh",
    "declining_performance": "Investigate Decline",
    "stale_content_review": "Review Freshness",
    "high_visibility_monitor": "Monitor",
    "low_priority_monitor": "Monitor"
}

queue["action"] = queue["reason_code"].map(action_map)

# -----------------------------
# 5. Rank the queue
# -----------------------------

queue = queue.sort_values(
    by=[
        "baseline_score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# -----------------------------
# 6. Keep the useful output columns
# -----------------------------

ranked_queue = queue[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "days_since_last_update",
        "trend_direction",
        "trend_pct"
    ]
]

print("Impressions threshold:", round(impressions_threshold, 2))
print("Days-since-update threshold:", round(update_age_threshold, 2))
print("CTR threshold:", round(ctr_threshold, 4))

print("\nRanked queue created.")
print("Rows:", len(ranked_queue))

display(ranked_queue.head(20))

Impressions threshold: 3615.25
Days-since-update threshold: 104.0
CTR threshold: 0.07

Ranked queue created.
Rows: 30000


,rank,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update,trend_direction,trend_pct
0,1,content_813e88069237,5,high_priority_refresh,Review and Refresh,233561,0.06,104,down,-33.8
1,2,content_c8e9d6ab9013,5,high_priority_refresh,Review and Refresh,208678,0.00,104,down,-43.4
2,3,content_8b36799b7e44,5,high_priority_refresh,Review and Refresh,141400,0.02,104,down,-62.7
3,4,content_c1fe78bc4e37,5,high_priority_refresh,Review and Refresh,134055,0.03,104,down,-41.4
4,5,content_e752a4e03dd3,5,high_priority_refresh,Review and Refresh,130892,0.01,104,down,-52.7
5,6,content_54baba704595,5,high_priority_refresh,Review and Refresh,130617,0.01,104,down,-54.8
6,7,content_124763d39ca5,5,high_priority_refresh,Review and Refresh,129803,0.01,104,down,-73.1
7,8,content_15bbc0978284,5,high_priority_refresh,Review and Refresh,109577,0.04,104,down,-53.9
8,9,content_40c50ec4c06e,5,high_priority_refresh,Review and Refresh,90972,0.06,104,down,-52.9
9,10,content_f8de7d4cee60,5,high_priority_refresh,Review and Refresh,89803,0.02,104,down,-37.0


In [ ]:
# Section 1 validation — inspect the action queue

print("Action distribution:")
display(
    ranked_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="rows")
)

print("\nReason-code distribution:")
display(
    ranked_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="rows")
)

print("\nScore distribution:")
display(
    ranked_queue["baseline_score"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("score")
    .reset_index(name="rows")
)

print("\nTop-20 action summary:")
display(
    ranked_queue.head(20)[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

# Add a simple confidence note for human review

def confidence_note(row):
    if row["baseline_score"] >= 4:
        return "Higher priority based on multiple observed signals; human review required."
    elif row["baseline_score"] >= 2:
        return "Directional priority based on some observed signals; verify context before acting."
    else:
        return "Low-priority signal; insufficient evidence for active intervention."

ranked_queue["confidence_note"] = queue.apply(
    confidence_note,
    axis=1
)

print("Section 1 queue with confidence notes:")
display(
    ranked_queue.head(20)
)

Action distribution:


,action,rows
0,Investigate Decline,10296
1,Monitor,10064
2,Review and Refresh,6109
3,Review Freshness,3373
4,Review Title / Snippet,158



Reason-code distribution:


,reason_code,rows
0,declining_performance,10296
1,low_priority_monitor,8312
2,declining_stale_content,3752
3,stale_content_review,3373
4,high_priority_refresh,2357
5,high_visibility_monitor,1752
6,high_visibility_low_ctr,158



Score distribution:


...


Top-20 action summary:


,rank,baseline_score,reason_code,action
0,1,5,high_priority_refresh,Review and Refresh
1,2,5,high_priority_refresh,Review and Refresh
2,3,5,high_priority_refresh,Review and Refresh
3,4,5,high_priority_refresh,Review and Refresh
4,5,5,high_priority_refresh,Review and Refresh
5,6,5,high_priority_refresh,Review and Refresh
6,7,5,high_priority_refresh,Review and Refresh
7,8,5,high_priority_refresh,Review and Refresh
8,9,5,high_priority_refresh,Review and Refresh
9,10,5,high_priority_refresh,Review and Refresh


Section 1 queue with confidence notes:


/tmp/ipykernel_544/806513714.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranked_queue["confidence_note"] = queue.apply(


,rank,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update,trend_direction,trend_pct,confidence_note
0,1,content_813e88069237,5,high_priority_refresh,Review and Refresh,233561,0.06,104,down,-33.8,Higher priority based on multiple observed sig...
1,2,content_c8e9d6ab9013,5,high_priority_refresh,Review and Refresh,208678,0.00,104,down,-43.4,Higher priority based on multiple observed sig...
2,3,content_8b36799b7e44,5,high_priority_refresh,Review and Refresh,141400,0.02,104,down,-62.7,Higher priority based on multiple observed sig...
3,4,content_c1fe78bc4e37,5,high_priority_refresh,Review and Refresh,134055,0.03,104,down,-41.4,Higher priority based on multiple observed sig...
4,5,content_e752a4e03dd3,5,high_priority_refresh,Review and Refresh,130892,0.01,104,down,-52.7,Higher priority based on multiple observed sig...
5,6,content_54baba704595,5,high_priority_refresh,Review and Refresh,130617,0.01,104,down,-54.8,Higher priority based on multiple observed sig...
6,7,content_124763d39ca5,5,high_priority_refresh,Review and Refresh,129803,0.01,104,down,-73.1,Higher priority based on multiple observed sig...
7,8,content_15bbc0978284,5,high_priority_refresh,Review and Refresh,109577,0.04,104,down,-53.9,Higher priority based on multiple observed sig...
8,9,content_40c50ec4c06e,5,high_priority_refresh,Review and Refresh,90972,0.06,104,down,-52.9,Higher priority based on multiple observed sig...
9,10,content_f8de7d4cee60,5,high_priority_refresh,Review and Refresh,89803,0.02,104,down,-37.0,Higher priority based on multiple observed sig...


###2. Intended use and limits

This playbook is intended for content and SEO reviewers who need to prioritize pages for human review.

The ranked queue provides decision-support by combining observable signals such as recent performance direction, impressions, click-through rate, and content freshness. It helps reviewers decide which pages may deserve earlier investigation or review.

The ranking does not establish that a particular action will improve performance. A declining page may have causes that are not represented in this dataset, and a high-priority page may not benefit from a refresh.

The queue should therefore be used for prioritization, not automatic publishing, rewriting, deletion, or other irreversible content changes.

The recommendations are based on the available dataset and its measurement windows. They should not be treated as universal thresholds or as evidence of causal effects. Human reviewers should consider search intent, content quality, business context, technical issues, and other information not captured by the dataset before acting.

In [ ]:
# Section 2 — Intended-use population summary

print("Playbook population:", len(ranked_queue))

print("\nAction counts:")
print(
    ranked_queue["action"]
    .value_counts()
)

print("\nHighest priority score:", ranked_queue["baseline_score"].max())

print(
    "Rows with score >= 4:",
    (ranked_queue["baseline_score"] >= 4).sum()
)

print(
    "Rows with score >= 3:",
    (ranked_queue["baseline_score"] >= 3).sum()
)

print(
    "Rows assigned an active review/intervention action:",
    ranked_queue["action"].isin([
        "Review and Refresh",
        "Review Freshness",
        "Review Title / Snippet",
        "Investigate Decline"
    ]).sum()
)

Playbook population: 30000

Action counts:
action
Investigate Decline       10296
Monitor                   10064
Review and Refresh         6109
Review Freshness           3373
Review Title / Snippet      158
Name: count, dtype: int64

Highest priority score: 5
Rows with score >= 4: 2357
Rows with score >= 3: 5748
Rows assigned an active review/intervention action: 19936


###3. Human review + the no-go list

A human reviewer must check the page context before taking action. The reviewer should verify the search intent, current content quality, relevance, freshness, and whether there are technical or measurement issues that could explain the observed signal.

For pages marked for refresh, the reviewer should confirm that the content is actually outdated or incomplete and identify what should change before any edit is made.

For pages marked for title or snippet review, the reviewer should confirm that the page is relevant to the observed search demand and that weak click-through is not explained by another factor.

For pages marked as investigate decline, the reviewer should first determine whether the decline is meaningful and whether there is an identifiable reason to intervene.

The following should not be automated:

- Publishing or replacing content without human approval.
- Deleting or pruning a page solely because it receives a low score.
- Rewriting titles, descriptions, or content solely from the queue.
- Treating a declining trend as proof that a refresh will recover performance.
- Making irreversible content or business decisions from the ranking alone.
- Treating the score as a probability of success.
- Ignoring search intent, content quality, technical issues, or business context because a page received a high priority score.

The queue is therefore a prioritization tool. Final action remains with a human reviewer.

In [ ]:
# Section 3 — Human-review fields

review_queue = ranked_queue.copy()

review_queue["human_review_required"] = True

review_queue["review_checks"] = (
    "Check search intent, content quality, freshness, "
    "technical context, and business relevance before acting."
)

review_queue["automation_status"] = "Human approval required"

print("Human review required for all queue items:",
      review_queue["human_review_required"].all())

print("\nAutomation status:")
print(
    review_queue["automation_status"]
    .value_counts()
)

print("\nExample review queue:")
display(
    review_queue.head(10)[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "human_review_required",
            "automation_status"
        ]
    ]
)

Human review required for all queue items: True

Automation status:
automation_status
Human approval required    30000
Name: count, dtype: int64

Example review queue:


,rank,content_id,action,reason_code,confidence_note,human_review_required,automation_status
0,1,content_813e88069237,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
1,2,content_c8e9d6ab9013,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
2,3,content_8b36799b7e44,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
3,4,content_c1fe78bc4e37,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
4,5,content_e752a4e03dd3,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
5,6,content_54baba704595,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
6,7,content_124763d39ca5,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
7,8,content_15bbc0978284,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
8,9,content_40c50ec4c06e,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required
9,10,content_f8de7d4cee60,Review and Refresh,high_priority_refresh,Higher priority based on multiple observed sig...,True,Human approval required


## 4. Monitoring / retrain triggers

The playbook should be monitored over time because content performance and search behavior can change.

The following are proposed review or retrain triggers:

- **Performance drift:** Re-evaluate the model if measured validation performance declines materially compared with the Week-5 grouped validation result.
- **Label-rate drift:** Review the playbook if the observed rate of declining content changes substantially from the current measured rate.
- **Feature drift:** Monitor major input features such as impressions, clicks, CTR, average position, and content age for substantial changes in their distributions.
- **Action-queue drift:** Review the rules if the proportion of content receiving high-priority or intervention actions changes substantially without an understood business reason.
- **Relationship drift:** Re-evaluate the model if the observed relationship between the input features and declining performance changes over time.
- **Data-quality problems:** Stop or review the queue if required features have unexpected missing values, invalid ranges, or changed definitions.

These are monitoring triggers rather than proven production thresholds. A trigger should lead to human investigation before retraining or changing the playbook.

Retraining should be considered when measured model performance or feature/label relationships have changed enough that the existing model no longer provides reliable decision-support.

In [ ]:
# Section 4 — Current monitoring reference values

monitoring_reference = {
    "population_rows": len(queue),
    "declining_rate": round((df["trend_direction"] == "down").mean(), 4),
    "high_priority_rows_score_5": int((queue["baseline_score"] == 5).sum()),
    "high_priority_rows_score_4_or_5": int((queue["baseline_score"] >= 4).sum()),
    "active_review_or_intervention_rows": int(
        queue["action"].isin([
            "Investigate Decline",
            "Review and Refresh",
            "Review Freshness",
            "Review Title / Snippet"
        ]).sum()
    ),
}

print("Current monitoring reference values")
print("----------------------------------")

for key, value in monitoring_reference.items():
    print(f"{key}: {value}")

Current monitoring reference values
----------------------------------
population_rows: 30000
declining_rate: 0.5421
high_priority_rows_score_5: 421
high_priority_rows_score_4_or_5: 2357
active_review_or_intervention_rows: 19936


## 5. Exports for the paper

The ranked action queue is exported to `work/outputs/` so the paper can reuse the same decision-support output generated by this notebook.

The export contains the content identifier, priority score, reason code, recommended action, observed performance fields, confidence note, and human-review status.

The queue is intended for human review and decision support; it is not an automated production action list.

In [ ]:
from pathlib import Path

# Create the output directory if it does not already exist
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the complete ranked queue
queue_path = output_dir / "ml10_ranked_action_queue.csv"
ranked_queue.to_csv(queue_path, index=False)

print("Queue exported successfully.")
print("Path:", queue_path)
print("Rows:", len(ranked_queue))
print("Columns:", len(ranked_queue.columns))

Queue exported successfully.
Path: work/outputs/ml10_ranked_action_queue.csv
Rows: 30000
Columns: 11


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.